# Tentativo 19 · 7. I risultati in una pagina

Dataset anonimo, 226 testate, uno split per voto, 5 fold; ogni misura è per testata, con l'intervallo al 95% (bootstrap
sulle testate). Le tabelle vengono dal notebook 6 (`Results/`), che spiega come sono fatte.

- **Spazio dei livelli**: il livello NewsGuard l1–l5. Il classificatore vota il livello; il regressore ci arriva con i
  confini NewsGuard, con le quote del training o con il voto degli articoli.
- **Spazio del voto**: il voto 0–100. Il regressore fa la media; il classificatore ci arriva con il voto atteso.
- **Nessun testo**: chi risponde sempre la classe più frequente o la media del training.

Colori: più scuro = meglio (per MAE ed errore, più chiaro = meglio).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OUT = "Results/"
tutte = pd.read_csv(OUT + "tutte_le_misure.csv")
ic = pd.read_csv(OUT + "intervalli.csv")
coppie = pd.read_csv(OUT + "classificatore_contro_regressore.csv")
pd.set_option("display.width", 220)


def con_intervallo(spazio, misure):
    """Una tabella modello × vista con «valore [da; a]» per le misure scelte."""
    d = ic[ic.spazio == spazio].copy()
    d["testo"] = [f"{v:.3f} [{a:.3f}; {b:.3f}]" if abs(v) < 2 else f"{v:.1f} [{a:.1f}; {b:.1f}]"
                  for v, a, b in zip(d.valore, d.da, d.a)]
    return d.pivot_table(index=["modello", "filosofia", "vista"], columns="misura", values="testo", aggfunc="first")[misure]


def colori(spazio, misure, al_contrario=()):
    """La stessa tabella con i soli valori, colorata colonna per colonna."""
    d = tutte[tutte.spazio == spazio].set_index(["modello", "filosofia", "vista"])[misure]
    try:
        st = d.style.format("{:.3f}")
    except AttributeError:                                  # senza jinja2 niente colori, solo i numeri
        return d.round(3)
    for c in misure:
        st = st.background_gradient(cmap="Blues_r" if c in al_contrario else "Blues", subset=[c])
    return st

## 1. Spazio dei livelli

In [ ]:
LIV = ["F1 macro", "tolerant (±1 livello)", "kappa quadratica", "Spearman", "affidabile: accuratezza"]
colori("livelli", LIV + ["accuratezza", "errore medio in livelli"], al_contrario=["errore medio in livelli"])

In [ ]:
con_intervallo("livelli", LIV)

## 2. Spazio del voto

In [ ]:
VOT = ["MAE", "Spearman", "entro 10 punti", "entro 20 punti", "affidabile: accuratezza"]
colori("voto", VOT + ["R²", "CCC di Lin"], al_contrario=["MAE"])

In [ ]:
con_intervallo("voto", VOT)

## 3. Classificatore o regressore?
Regressore − classificatore, stesso modello, stesse testate. ✓ = l'intervallo non contiene lo 0. Per MAE ed errore,
negativo = meglio il regressore; per tutte le altre misure, positivo = meglio il regressore.

In [ ]:
c = coppie.copy()
c["regressore − classificatore"] = [f"{d:+.3f} [{a:+.3f}; {b:+.3f}]{' ✓' if n else ''}" for d, a, b, n in
                                    zip(c.differenza, c.da, c.a, c["non è caso"])]
c.pivot_table(index=["spazio", "misura", "regressore"], columns="modello", values="regressore − classificatore", aggfunc="first")

## 4. In un grafico

In [ ]:
RIGHE = [("livelli", ["F1 macro", "tolerant (±1 livello)"]), ("voto", ["MAE", "Spearman"])]
COLORE = {"classificatore": "#2a78d6", "regressore": "#eb6834", "riferimento": "#8a8983"}
n = [ic[ic.spazio == s].groupby(["modello", "vista"]).ngroups for s, _ in RIGHE]
fig, assi = plt.subplots(2, 2, figsize=(13, 0.32 * sum(n) + 2), gridspec_kw={"height_ratios": n})
for riga, (spazio, misure) in enumerate(RIGHE):
    for col, misura in enumerate(misure):
        ax = assi[riga, col]
        d = ic[(ic.spazio == spazio) & (ic.misura == misura)].reset_index(drop=True)
        y = np.arange(len(d))
        ax.errorbar(d.valore, y, xerr=[d.valore - d.da, d.a - d.valore], fmt="none", ecolor="#b5b3ab", lw=1.5, capsize=2)
        ax.scatter(d.valore, y, s=40, c=[COLORE[f] for f in d.filosofia], zorder=3)
        ax.set_yticks(y, [f"{m} · {v}" for m, v in zip(d.modello, d.vista)] if col == 0 else [""] * len(y), fontsize=8)
        ax.invert_yaxis(); ax.set_title(f"{spazio}: {misura}", fontsize=10, loc="left", fontweight="bold")
        ax.grid(axis="x", color="#e1e0d9"); ax.spines[["top", "right"]].set_visible(False)
for f, c in COLORE.items():
    assi[0, 0].scatter([], [], c=c, label=f)
fig.legend(loc="upper center", ncol=3, fontsize=9, frameon=False, bbox_to_anchor=(0.5, 1.03))
plt.tight_layout(); plt.savefig(OUT + "risultati.png", dpi=150, bbox_inches="tight"); plt.show()

## 5. In breve (generato dalle tabelle)

In [ ]:
def migliore(spazio, misura, minore=False):
    d = tutte[(tutte.spazio == spazio) & (tutte.filosofia != "riferimento")]
    r = d.loc[d[misura].idxmin() if minore else d[misura].idxmax()]
    rif = tutte[(tutte.spazio == spazio) & (tutte.filosofia == "riferimento")][misura].iloc[0]
    return f"- **{spazio}, {misura}**: {r.modello} ({r.filosofia}, {r.vista}) = {r[misura]:.3f}; senza testo {rif:.3f}"


print("\n".join([migliore("livelli", "F1 macro"), migliore("livelli", "tolerant (±1 livello)"),
                  migliore("livelli", "Spearman"), migliore("voto", "MAE", minore=True), migliore("voto", "Spearman"),
                  migliore("voto", "entro 10 punti")]))
vinte = coppie[coppie["non è caso"]]
print(f"\nConfronti classificatore/regressore con una differenza che non è caso: {len(vinte)} su {len(coppie)}")